# Phase 1.4 : Entraînement et diagnostic overfitting

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase1_3_architecture.ipynb`.

**Ce notebook est autonome** : la section "Reprise" ci-dessous refait le setup des phases 1.1 à 1.3 (téléchargement, organisation, pipeline de données, architecture du CNN), sans les explications déjà vues.

## Reprise (setup phases 1.1 à 1.3, condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ne suffisent : ce dataset
    # contient quelques fichiers que seul le decodeur TensorFlow rejette au moment de
    # l'entrainement (InvalidArgumentError "Unknown image file format"). On teste donc
    # directement avec tf.io.decode_image, le meme decodeur utilise par
    # image_dataset_from_directory en phase 1.2.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            tf.io.decode_image(img_bytes, channels=3)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)
# Filet de securite : le decodeur d'images en mode graphe (utilise ici) est parfois plus
# strict que tf.io.decode_image en mode eager (utilise a l'organisation des dossiers).
# ignore_errors() saute silencieusement tout exemple qui ferait planter le decodeur.
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models


def build_cnn_scratch(input_shape):
    """CNN from scratch pour la classification binaire.
    Architecture délibérément simple : on veut voir l'overfitting, pas l'éviter.
    """
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model


model_scratch = build_cnn_scratch(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_scratch.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

## Phase 1.4 : Entraînement et diagnostic overfitting

**Objectif** : entraîner le CNN from scratch pendant 20 epochs, lancer TensorBoard, lire les courbes train/val, et identifier le point de divergence (overfitting).

C'est voulu : ce modèle est délibérément simple, sans augmentation ni dropout — on veut *voir* l'overfitting avant de le corriger en phase 2.

In [ ]:
import datetime, time

# Callback TensorBoard : un log par run, avec un timestamp pour ne pas écraser les précédents.
log_dir = "logs/scratch/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# EarlyStopping : on coupe quand la val_loss ne s'ameliore plus pendant 5 epochs,
# et on restaure les meilleurs poids (pas ceux du dernier epoch, potentiellement surappris).
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

In [ ]:
start = time.time()

history_scratch = model_scratch.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[tensorboard_callback, early_stopping],
)

training_time_scratch = time.time() - start
print(f"Temps d'entrainement : {training_time_scratch:.0f}s")
print(f"val_accuracy finale : {max(history_scratch.history['val_accuracy']):.3f}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/scratch

In [ ]:
import matplotlib.pyplot as plt


def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history["loss"], label="Train loss")
    ax1.plot(history.history["val_loss"], label="Val loss")
    ax1.set_title(f"{title} - Loss")
    ax1.legend()

    ax2.plot(history.history["accuracy"], label="Train accuracy")
    ax2.plot(history.history["val_accuracy"], label="Val accuracy")
    ax2.set_title(f"{title} - Accuracy")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f"curves_{title.lower().replace(' ', '_')}.png", dpi=100)
    plt.show()


plot_history(history_scratch, "CNN scratch")

### Ce que tu dois observer

La `val_accuracy` plafonne (autour de 65-72% typiquement pour ce dataset) pendant que la `train_accuracy` continue de monter. La `val_loss` remonte après quelques epochs. **C'est l'overfitting classique, et c'est voulu** : ce CNN mémorise le train set sans généraliser correctement — normal, il n'a ni augmentation ni dropout.

Identifie sur les courbes : à quel epoch la `val_loss` commence-t-elle à remonter ? C'est le point que `EarlyStopping` doit attraper (`restore_best_weights=True` te ramène à ce point automatiquement, pas au dernier epoch).

### Avant de passer à la phase 2.1

- **Happy path** : `train_accuracy` monte vers 80-90%, `val_accuracy` plafonne en dessous (souvent 65-75%), gap visible dès epoch 8-12, `val_loss` qui remonte. TensorBoard ouvert et synchrone avec les logs.
- **Edge case** : relancer avec `BATCH_SIZE=4` au lieu de 32. Le bruit du gradient explose, les courbes oscillent violemment, la `val_accuracy` finale est probablement plus basse. Comparer l'écart.
- **Adversarial** : si `shuffle=True` avait été oublié dans `image_dataset_from_directory` (phase 1.2), les batches contiendraient des images d'une seule classe d'affilée — la loss oscillerait sans jamais se stabiliser. On l'a déjà vérifié, donc ce n'est pas un risque ici.

**Prochaine étape** : Phase 2.1 — pipeline de data augmentation, pour commencer à corriger cet overfitting.